# Movie Recommendation System — Demo Notebook

**Implementation of Machine Learning Ensemble Model for Movie Recommendation**

This notebook demonstrates the complete pipeline:
1. Load and merge datasets
2. Parse metadata and create tags
3. Build TF-IDF matrix
4. Train ML models
5. Compute cosine similarity
6. Generate recommendations using ensemble voting
7. Batman example

## 1. Imports

In [ ]:
import sys
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import cosine_similarity

# Add project root to path so we can import src modules
PROJECT_ROOT = Path('.').resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import extract_names, extract_cast_names

print('All imports successful!')

## 2. Load Datasets

In [ ]:
DATA_DIR = PROJECT_ROOT / 'data'

movies = pd.read_csv(DATA_DIR / 'tmdb_5000_movies.csv')
credits = pd.read_csv(DATA_DIR / 'tmdb_5000_credits.csv')

# Strip column-name whitespace
movies.columns = movies.columns.str.strip()
credits.columns = credits.columns.str.strip()

print(f'Movies shape : {movies.shape}')
print(f'Credits shape: {credits.shape}')
movies.head(2)

## 3. Merge Datasets

In [ ]:
# Rename 'id' to 'movie_id' in movies
if 'id' in movies.columns and 'movie_id' not in movies.columns:
    movies = movies.rename(columns={'id': 'movie_id'})

# Merge on movie_id
credits_to_merge = credits.drop(columns=['title'], errors='ignore')
df = pd.merge(movies, credits_to_merge, on='movie_id', how='inner')

print(f'Merged shape: {df.shape}')
df[['movie_id', 'title', 'genres', 'overview']].head()

## 4. Create Tags

In [ ]:
# Parse metadata
df['parsed_genres'] = df['genres'].apply(extract_names)
df['parsed_keywords'] = df['keywords'].apply(extract_names)
df['parsed_cast'] = df['cast'].apply(extract_cast_names)

# Fill missing overview
df['overview'] = df['overview'].fillna('')

# Combine into tags
df['tags'] = (
    df['overview'].astype(str) + ' ' +
    df['parsed_genres'].astype(str) + ' ' +
    df['parsed_keywords'].astype(str) + ' ' +
    df['parsed_cast'].astype(str)
)

# Normalise
df['tags'] = df['tags'].str.lower().str.strip()
df['tags'] = df['tags'].str.replace(r'\s+', ' ', regex=True)
df['tags'] = df['tags'].fillna('')

print('Sample tags for Batman:')
batman_mask = df['title'].str.lower() == 'batman'
if batman_mask.any():
    print(df.loc[batman_mask, 'tags'].values[0][:200] + '...')
else:
    print('Batman not found in dataset.')

## 5. TF-IDF Vectorization

In [ ]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['tags'])

print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'  Movies  : {tfidf_matrix.shape[0]}')
print(f'  Features: {tfidf_matrix.shape[1]}')

## 6. Train/Test Split

In [ ]:
# NOTE: The original project treats movie_id as the supervised target
# for the auxiliary classifiers. This is a project-specific formulation
# and should not be interpreted as conventional recommendation-system
# ground truth.

labels = df['movie_id']
X_train, X_test, y_train, y_test = train_test_split(
    tfidf_matrix, labels, test_size=0.2, random_state=42
)

print(f'Training samples: {X_train.shape[0]}')
print(f'Testing samples : {X_test.shape[0]}')

## 7. Model Initialisation & Training

In [ ]:
# SVM
print('Training SVM ...')
svm_model = SVC(kernel='linear', probability=True)
svm_model.fit(X_train, y_train)
print('  SVM done.')

# Decision Tree
print('Training Decision Tree ...')
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
print('  Decision Tree done.')

# KNN
print('Training KNN ...')
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)
print('  KNN done.')

# Gradient Boosting
print('Training Gradient Boosting ...')
gb_model = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_model.fit(X_train, y_train)
print('  Gradient Boosting done.')

## 8. Model Evaluation

In [ ]:
for name, model in [('SVM', svm_model), ('Decision Tree', dt_model),
                     ('KNN', knn_model), ('Gradient Boosting', gb_model)]:
    try:
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        print(f'{name:25s} accuracy: {acc:.4f}')
    except Exception as e:
        print(f'{name:25s} accuracy: N/A ({e})')

## 9. Cosine Similarity

In [ ]:
cosine_sim = cosine_similarity(tfidf_matrix)
print(f'Cosine similarity matrix shape: {cosine_sim.shape}')

## 10. Recommendation Function

In [ ]:
def get_recommendations(movie_title, n=10):
    """Ensemble recommendations: cosine similarity + KNN + Counter voting."""
    # Find movie index (case-insensitive)
    lower_titles = df['title'].str.lower()
    matches = df.index[lower_titles == movie_title.lower()]
    if len(matches) == 0:
        return f"Movie '{movie_title}' not found in dataset."
    idx = matches[0]
    
    # Cosine similarity recommendations
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:n+1]  # skip self
    cosine_indices = [i for i, _ in sim_scores]
    
    # KNN recommendations
    distances, indices = knn_model.kneighbors(
        tfidf_matrix[idx], n_neighbors=n+1
    )
    knn_indices = [i for i in indices.flatten() if i != idx][:n]
    
    # Combine and vote
    combined = cosine_indices + knn_indices
    counter = Counter(combined)
    counter.pop(idx, None)  # safety: remove input movie
    top = [i for i, _ in counter.most_common(n)]
    
    return df.loc[top, 'title'].tolist()

## 11. Batman Demonstration

In [ ]:
recommendations = get_recommendations('Batman', n=10)

print('='*50)
print("Recommended Movies similar to 'Batman':")
print('='*50)
if isinstance(recommendations, str):
    print(recommendations)
else:
    for i, title in enumerate(recommendations, 1):
        print(f'  {i:2d}. {title}')